### Temporal split and deterministic candidates

Evaluation uses one held-out positive plus 100 unobserved negatives. Items unseen in train are excluded from pure collaborative-filtering evaluation.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import torch

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
PROCESSED = ROOT / "data" / "processed"
MODELS = ROOT / "models"
RESULTS = ROOT / "results"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)


In [ ]:
from src.data.implicit import build_user_history, build_eval_candidates, load_movielens_ratings

raw = load_movielens_ratings(ROOT / "data/raw/ml-1m/ratings.dat")
train = pd.read_csv(PROCESSED / "train.csv")
validation = pd.read_csv(PROCESSED / "validation.csv")
test = pd.read_csv(PROCESSED / "test.csv")
users = pd.read_csv(PROCESSED / "user_mapping.csv")
movies = pd.read_csv(PROCESSED / "movie_mapping.csv")
user_map = dict(zip(users.user_id.astype(int), users.user_idx.astype(int)))
movie_map = dict(zip(movies.movie_id.astype(int), movies.movie_idx.astype(int)))
for frame in (train, validation, test):
    frame["user_idx"] = frame.user_id.map(user_map)
    frame["movie_idx"] = frame.movie_id.map(movie_map)
history = build_user_history(raw)
train_movies = sorted(train.movie_id.astype(int).unique())


In [ ]:
for name, frame, seed in (("validation", validation, 43), ("test", test, 44)):
    candidates = build_eval_candidates(frame.dropna(), history, train_movies, movie_map, 100, seed)
    candidates.to_csv(PROCESSED / f"{name}_candidates_100neg.csv", index=False)
    print(name, candidates.shape, "users:", candidates.user_id.nunique())


In [ ]:
validation_candidates = pd.read_csv(PROCESSED / "validation_candidates_100neg.csv")
test_candidates = pd.read_csv(PROCESSED / "test_candidates_100neg.csv")
print("validation candidate size:", validation_candidates.groupby("user_id").size().unique())
print("test candidate size:", test_candidates.groupby("user_id").size().unique())
